# 03 — Treasury-Pair Relationships

**Purpose:** Establish whether ZT–ZF, ZF–ZN, ZT–ZN, ZN–ZB relationships (curve segments 2s5s/5s10s/2s10s/10s30s) are stable and researchable (mandate §4.1).

**Research questions:**
1. How do price relationships map to curve moves (level vs slope)?
2. What are the dollar-volatility ratios and available DV01 approximations per pair?
3. Is there intraday seasonality (US data releases, auction times)?
4. How did pairs behave across the 2022-2023 tightening vs 2024+ easing; around FOMC/CPI?
5. Do CTD switches / delivery cycles contaminate the series (A-012)?

**Data used:** minute continuous-adjusted closes for ZT/ZF/ZN/ZB (**blocked**, L-001); FOMC/CPI calendar (to be sourced).


In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


## Methodology

Mirrors notebook 02 plus: dollar-vol ratio tables (`hedge_ratios.dollar_vol_ratio` inputs); approximate DV01 ratios where CTD data is available — otherwise empirical price-beta stands in and the approximation is logged; event-window behavior via `regimes.event_windows`; roll/delivery-month exclusion sensitivity.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


In [ ]:
if DATA_AVAILABLE:
    from spread_research.data_loader import load_local
    from spread_research.pair_builder import align_pair
    from spread_research.structural_breaks import rolling_correlation
    cfg = yaml.safe_load(open("../config/pair_definitions.yaml"))
    for name, p in cfg["treasury_pairs"].items():
        a = load_local(p["long_ref"], "minute", DATA_DIR)["close"]
        b = load_local(p["short_ref"], "minute", DATA_DIR)["close"]
        pair = align_pair(a, b)
        ra, rb = pair["a"].diff(), pair["b"].diff()
        print(name, "| n =", len(pair),
              "| corr(1d roll) median =", float(rolling_correlation(ra, rb, 390).median()))

## Results

**BLOCKED-ON-DATA** — this section intentionally contains no results. No synthetic or fabricated market findings are presented as evidence (CLAUDE.md gate 3). It will be populated when the notebook runs against real data.

## Limitations

Without CTD-level data, DV01 neutrality is approximate (L-006); delivery-month microstructure may require excluding more data than the equity book.

## Decision

Curve-pair viability verdicts feed notebooks 04/05; UB remains excluded (D-002).

## What this means for the algorithm

Treasury hedge ratios are expected to be materially unequal (e.g. several ZT per ZN); contract rounding on small size may dominate — quantified in notebook 12.